### Hyperparameter Tuning using Optuna

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [2]:
class MyNN(nn.Module):

    def __init__(self, input_dim, output_dim, num_hidden_layers, num_neurons_per_layer, dropout_rate):

        super().__init__()

        layers = [] 

        for i in range(num_hidden_layers):

            layers.append(nn.Linear(input_dim, num_neurons_per_layer))
            layers.append(nn.BatchNorm1d(num_neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = num_neurons_per_layer

        # create the output layer 
        layers.append(nn.Linear(num_neurons_per_layer, output_dim))

        self.model = nn.Sequential(*layers)    

    def forward(self, x):
        return self.model(x)
    
    

In [3]:
# load the dataset and create train_dataset and test_dataset 
df = pd.read_csv('./data/fmnist_small.csv')
X = df.iloc[:, 1:].values
y = df.iloc[:,0].values
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
# scaling on the features 
X_train = X_train / 255.0 
X_test = X_test / 255.0 

class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]    
    
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)    

In [4]:
def objective(trial):
    num_hidden_layers = trial.suggest_int("num_hidden_layers",1,8) # hidden layers starting from 1 to 8 
    num_hidden_neurons = trial.suggest_int("num_hidden_neurons",8,128,step=8)
    epochs = trial.suggest_int("epochs",10,100,step=10)
    learning_rate = trial.suggest_float("learning_rate",1e-10,1e-1,log=True)
    dropout_rate = trial.suggest_float("dropout_rate",0.1,0.8,step=0.1)
    batch_size = trial.suggest_categorical("batch_size",[16,32,64,128])
    optimizer_name = trial.suggest_categorical("optimizer_name",["Adam","SGD","RMSprop"])
    # L2 regularization 
    weight_decay = trial.suggest_float("weight_decay",1e-5,1e-3,log=True)


    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

    
    input_dimension = X_train.shape[1]
    output_dimension = 10
    
    model = MyNN(input_dim=input_dimension,output_dim=output_dimension,num_hidden_layers=num_hidden_layers,num_neurons_per_layer=num_hidden_neurons,dropout_rate=dropout_rate)

    optimizer = None 

    if optimizer_name == "Adam":
       optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else: 
        optimizer = optim.RMSprop(model.parameters(), lr = learning_rate, weight_decay=weight_decay)   

    criterion = nn.CrossEntropyLoss()         


    # training loop 
    for epoch in range(epochs):
        for batch_features, batch_labels in train_loader:
             output = model(batch_features)

             loss = criterion(output, batch_labels)

             optimizer.zero_grad()
             loss.backward()

             optimizer.step()

    model.eval() 

    total = 0
    correct = 0

    with torch.no_grad():

        for batch_features, batch_labels in test_loader:

            output = model(batch_features)

            _, predicted = torch.max(output,1) 

            total = total + batch_labels.shape[0]

            correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct / total

    return accuracy                  






In [5]:
import optuna 

study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=10)

ModuleNotFoundError: No module named 'optuna'

In [6]:
! pip install optuna

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for optuna from https://files.pythonhosted.org/packages/ac/24/7c731839566d30dc70556d9824ef17692d896c15e3df627bce8c16f753e1/optuna-4.8.0-py3-none-any.whl.metadata
  Obtaining dependency information for alembic>=1.5.0 from https://files.pythonhosted.org/packages/d2/29/6533c317b74f707ea28f8d633734dbda2119bbadfc61b2f3640ba835d0f7/alembic-1.18.4-py3-none-any.whl.metadata
  Obtaining dependency information for colorlog from https://files.pythonhosted.org/packages/6d/c1/e419ef3723a074172b68aaa89c9f3de486ed4c2399e2dbd8113a4fdcaf9e/colorlog-6.10.1-py3-none-any.whl.metadata
  Obtaining dependency information for sqlalchemy>=1.4.2 from https://files.pythonhosted.org/packages/c4/59/55a6d627d04b6ebb290693681d7683c7da001eddf90b60cfcc41ee907978/sqlalchemy-2.0.49-cp311-cp311-win_amd64.whl.metadata
  Using cached sqlalchemy-2.0.49-cp311-cp311-win_amd64.whl.metadata (9.8 kB)
  Obtaining depen


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import optuna 

study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=10)

C:\Users\92324\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-07 13:12:07,886] A new study created in memory with name: no-name-9f2593cf-efbe-4332-8d7f-d1a194542de3
C:\Users\92324\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[I 2026-05-07 13:12:46,237] Trial 0 finished with value: 0.0625 and parameters: {'num_hidden_layers': 4, 'num_hidden_neurons': 32, 'epochs': 100, 'learning_rate': 8.941871446872888e-10, 'dropout_rate': 0.30000000000000004, 'batch_size': 128, 'optimizer_name': 'RMSprop', 'weight_decay': 2.91273436722671e-05}. Best is trial 0 with value: 0.0625.
[I 2026-05-07 13:12:59,100] 

In [8]:
print("best value:",study.best_value)
print("best params", study.best_params)

best value: 0.8441666666666666
best params {'num_hidden_layers': 4, 'num_hidden_neurons': 56, 'epochs': 60, 'learning_rate': 0.032617393396295394, 'dropout_rate': 0.2, 'batch_size': 32, 'optimizer_name': 'SGD', 'weight_decay': 1.831093913406637e-05}
